# `05_rag/test.ipynb`

1. `langgraph.json` 생성 -> `langgraph dev` 명령어로 서버 up
2. 코드 테스트는 notebook 파일에서 진행 -> 실제 파이썬 모듈로 조립

## Custom RAG Agent
- RAG agent 를 langgraph 로 처음부터 만든다
- langchain 에서 제공하는 기본 agent가 아닌, 직접 원하는대로 기능을 구현할 수 있음

![image](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/langgraph-hybrid-rag-tutorial.png?fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=855348219691485642b22a1419939ea7)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### RAG 전처리
1. Load
2. Split
3. Embed
4. Store

In [11]:
from bs4.filter import SoupStrainer  # uv pip install beautifulsoup4
from langchain_community.document_loaders import WebBaseLoader

# 해당 블로그에서 우리가 원하는 부분만 골라내기
bs4_strainer = SoupStrainer(class_=('post-title', 'post-header', 'post-content'))

# 인터넷 웹 페이지 여러개를 저장
urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2025-05-01-thinking/",
]

docs = [WebBaseLoader(web_path=url, bs_kwargs={'parse_only': bs4_strainer}).load() for url in urls]

docs_list = [item for sublist in docs for item in sublist]

docs_list

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2024-11-28-reward-hacking/'}, page_content='\n\n      Reward Hacking in Reinforcement Learning\n    \nDate: November 28, 2024  |  Estimated Reading Time: 37 min  |  Author: Lilian Weng\n\n\nReward hacking occurs when a reinforcement learning (RL) agent exploits flaws or ambiguities in the reward function to achieve high rewards, without genuinely learning or completing the intended task. Reward hacking exists because RL environments are often imperfect, and it is fundamentally challenging to accurately specify a reward function.\nWith the rise of language models generalizing to a broad spectrum of tasks and RLHF becomes a de facto method for alignment training, reward hacking in RL training of language models has become a critical practical challenge. Instances where the model learns to modify unit tests to pass coding tasks, or where responses contain biases that mimic a user’s preference, are pretty concerning and are 

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
chunks = text_splitter.split_documents(docs_list)

print(len(chunks))

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vectorstore = InMemoryVectorStore(embedding=embeddings)
ids = vectorstore.add_documents(chunks)

220


### RAG 검색기 생성

In [ ]:
from langchain.tools import tool

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

@tool
def retrieve_blog_posts(query: str) -> str:
    """AI, LLM에 관련된 정보를 블로글 글에서 검색 후 return"""
    docs = retriever.invoke(query)
    # docs 는 메타데이터를 포함한 리스트 -> LLM 은 내용만 들어있는 str 이면 충분하다.
    # 전처리 작업
    return '\n\n'.join([doc.page_content for doc in docs])


print(retrieve_blog_posts.invoke({'query': '보상 해킹의 종류'}))

Reward hacking (Amodei et al., 2016)
Reward corruption (Everitt et al., 2017)
Reward tampering (Everitt et al. 2019)
Specification gaming (Krakovna et al., 2020)
Objective robustness (Koch et al. 2021)
Goal misgeneralization (Langosco et al. 2022)
Reward misspecifications (Pan et al. 2022)

Why does Reward Hacking Exist?#
Goodhart’s Law states that “When a measure becomes a target, it ceases to be a good measure”. The intuition is that a good metric can become corrupted once significant pressure is applied to optimize it. It is challenging to specify a 100% accurate reward objective and any proxy suffers the risk of being hacked, as RL algorithm exploits any small imperfection in the reward function definition. Garrabrant (2017) categorized Goodhart’s law into 4 variants:

Regressional - selection for an imperfect proxy necessarily also selects for noise.
Extremal - the metric selection pushes the state distribution into a region of different data distribution.
Causal -  when there is 

### 쿼리 생성

In [ ]:
from langgraph.graph import MessagesState
from langchain.chat_models import init_chat_model

# 응답 생성용 모델 | temperature => 0에 가까울수록 늘 하던 답변만 생성 / 1과 가깝거나 그 이상이면 새로운 답변들이 추가
# 0 -> 정확/일관/사실기반 | 0.7 -> 일반적인 | 1.0 -> 창의적인
res_llm = init_chat_model('gpt-4.1', temperature=0)


def generate_query_or_respond(state: MessagesState):
    """llm이 사용자 대화를 기반으로 바로 답변을 생성하거나
    RAG를 위한 query 를 결정한다."""
    tllm = res_llm.bind_tools([retrieve_blog_posts])
    res = tllm.invoke(state['messages'])
    return {'messages': [res]}


In [39]:
generate_query_or_respond({
    'messages': [
        {'role': 'user', 'content': 'llm에서 보상 해킹이란 어떤건지 블로그 글 찾아봐'}
    ]
})

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 72, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_c2e1a65248', 'id': 'chatcmpl-DHKCQ1TdOFQrVTxJgFYUKDlugGNtV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cd052-83d7-7031-bf97-13a0e254041d-0', tool_calls=[{'name': 'retrieve_blog_posts', 'args': {'query': 'llm 보상 해킹'}, 'id': 'call_m8EZ44S6KBUwyXpDLKghBJo5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 20, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, TypedDict

GRADE_PROMPT = """너는 사용자 질문과 검색된 문서의 연관성을 확인하는 평가자야.
사용자 질문: \n\n {question} \n\n
검색된 문서: \n\n {docs} \n\n
만약 검색된 문서에 사용자 질문에 있는 키워드나, 의미론적으로 같은 내용이 있으면 연관성을 확인한거야.
최종적으로 'yes' or 'no' 로 질문과 문서의 연관성 점수를 내줘.
"""

# Structured Output 제작용
class GradeDocuments(BaseModel):
    """문서를 yes/no 로 질문과 연관있는지 평가"""
    binary_score: str = Field(description='연관성 점수: 연관 있으면 "yes", 연관 없으면 "no"')


grader_llm = init_chat_model('gpt-4.1', temperature=0)


# Router
def grade_document(state: MessagesState):
    """검색해온 문서가 사용자 질문과 관련이 있는지 체크 후 다음 노드 결정"""
    question = state['messages'][0].content
    docs = state['messages'][-1].content

    prompt = GRADE_PROMPT.format(question=question, docs=docs)
    ollm = grader_llm.with_structured_output(GradeDocuments)
    res = ollm.invoke(
        [
            {'role': 'user', 'content': prompt}
        ]
    )
    
    score = res.binary_score

    if score == 'yes':
        return 'generate_answer'
    else:
        return 'rewrite_question'


c:\Users\user\Desktop\gaida-2\05-langgraph\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeDocuments(binary_score='yes'), input_type=GradeDocuments])
  return self.__pydantic_serializer__.to_python(


GradeDocuments(binary_score='yes')

### Rewrite Question
- 사용자 질문을 그대로 활용하여 RAG 검색한 문서 chunk 결과가 질문과 무관할때
- 사용자의 질문을 해당 Node가 재작성(develop)하여 다시 검색하도록 함.

In [59]:
from langchain.messages import HumanMessage

REWRITE_QUESTION_PROMPT = """주어진 질문을 살펴보고 그 안에 담긴 의도/의미를 추론하라.
---
질문: {original_question}
---
더 나은 질문만 생성하시오.
"""

def rewrite_question(state: MessagesState):
    """사용자 질문을 재 작성"""
    messages = state['messages']
    # TODO: 나중에 전체 맥락을 이해하고 질문을 정제하는 것으로 바꾸기
    question = messages[0].content
    prompt = REWRITE_QUESTION_PROMPT.format(original_question=question)
    response = res_llm.invoke(prompt)
    return {'messages': [HumanMessage(content=response.content)]}



In [ ]:
# rewrite_question 노드 테스트용 코드

from langchain_core.messages import convert_to_messages

input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {"role": "tool", "content": "meow", "tool_call_id": "1"},
        ]
    )
}

response = rewrite_question(input)
print(response["messages"][-1].content)

Lilian Weng이 언급한 reward hacking의 유형에는 어떤 것들이 있으며, 각각의 예시는 무엇인가요?
